In [2]:
import os
import numpy as np
import pickle
import pandas as pd
from scipy.stats import zscore
from brainbox.io.one import SessionLoader
from sklearn.preprocessing import StandardScaler
import gc
import concurrent.futures

from segmentation_functions import get_left_licks, resample_common_time, interpolate_nans
from one.api import ONE
one = ONE(mode='remote')


In [1]:
prefix = '/home/ines/repositories/'
# prefix = '/Users/ineslaranjeira/Documents/Repositories/'

# Get training sessions

In [4]:
data_path = '/home/ines/repositories/representation_learning_variability/paper-individuality/segmentation/1_camera_setup/temp_data/design_matrices/' # data should be moved to the drive manually
data_path = '/home/ines/repositories/representation_learning_variability/paper-individuality/data/design_matrices/1_camera_setup/session_1_individuality/wheel/' # data should be moved to the drive manually
sessions = list(pd.read_csv('nm_filtered_eids.csv')['eid'])
sessions = list(pd.read_csv('lda_first_training_eids.csv')['eid'])

In [5]:
os.chdir(data_path)
files = os.listdir()
sessions_to_process = []

for s, sess in enumerate(sessions):
    file_path = one.eid2path(sess)

    if prefix == '/home/ines/repositories/':
        mouse_name = file_path.parts[8]
    else:
        mouse_name = file_path.parts[7]

    filename = "design_matrix_" + str(sess) + '_'  + mouse_name
    if filename not in files:
        sessions_to_process.append((sess))

len(sessions_to_process)

56

In [6]:
def process_design_matrix(session):

    file_path = one.eid2path(session)
    if prefix == '/home/ines/repositories/':
        mouse_name = file_path.parts[8]
    else:
        mouse_name = file_path.parts[7]

    """ LOAD VARIABLES """
    sl = SessionLoader(eid=session, one=one)
    try:
        sl.load_session_data(trials=True, wheel=True, motion_energy=False)

        # Check if all data is available
        if np.sum(sl.data_info['is_loaded']) >= 1:

            # Wheel
            wheel = sl.wheel
            wheel_t = np.asarray(wheel['times'], dtype=np.float64)
            wheel_vel = wheel['velocity'].astype(np.float32)

            # Common resampling window
            onset = wheel_t.min()
            offset = wheel_t.max()
            fs = 30
            ref_t = np.arange(onset, offset, 1 / fs, dtype=np.float64)

            # Restrict to time window
            def restrict(t, x):
                mask = (t >= onset) & (t <= offset)
                return t[mask], x[mask]

            wheel_t, wheel_vel = restrict(wheel_t, wheel_vel)

            # Resample
            wh_down, rt = resample_common_time(ref_t, wheel_t, wheel_vel, kind='linear')

            # Create design matrix
            design_matrix = pd.DataFrame({
                'Bin': rt,
                'avg_wheel_vel': wh_down
            })

            # """ LOAD TRIAL DATA """
            session_trials = sl.trials
            session_start = list(session_trials['goCueTrigger_times'])[0]
            design_matrix = design_matrix.loc[(design_matrix['Bin'] > session_start)]

            """ SAVE DATA """       
            # Save unnormalized design matrix
            filename = data_path + "design_matrix_" + str(session) + '_'  + mouse_name
            design_matrix.to_parquet(filename, compression='gzip')  

            # Save trials
            filename = data_path + "session_trials_" + str(session) + '_'  + mouse_name
            session_trials.to_parquet(filename, compression='gzip')  
            
            del design_matrix, session_trials, sl
            gc.collect()

        else:
            print('Data missing for session '+session)  
    except Exception as e:
        # We catch the exception, print the specific error, and move to the next session
        print(f"Failed to load session {session}. Error: {e}")


def parallel_process_data(sessions, function_name):
    with concurrent.futures.ThreadPoolExecutor() as executor:

        # Process each chunk in parallel
        executor.map(function_name, sessions)

In [ ]:
for s, session in enumerate(sessions_to_process):
    process_design_matrix(session)